In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import textwrap

# LOAD DATA
df = pd.read_csv("questionnaire_data_other_factors.csv")


def merge_groups(g):
    if g["n"].sum() == 0:
        return pd.Series({"association": 0, "n": 0})

    return pd.Series({
        "association": np.average(
            g["association"],
            weights=g["n"]
        ),
        "n": g["n"].sum()
    })


# Aggregate visits if multiple columns were merged
num_bin = df[df["type"].isin(["numeric", "binary"])].copy()

num_bin = (
    num_bin
    .groupby(
        ["score", "factor", "type"],
        as_index=False
    )
    .apply(
        merge_groups,
        include_groups=False
    )
    .reset_index()
)


cat = df[df["type"] == "categorical"].copy()

cat = cat[
    ~cat["factor"].isin([
        "pregnant",
        "shiftwork",
        "working_hours_model"
    ])
]
cat = (
    cat
    .groupby(
        ["score", "factor", "value"],
        as_index=False
    )
    .apply(
        merge_groups,
        include_groups=False
    )
    .reset_index()
)

cat["type"] = "categorical"


df = pd.concat(
    [num_bin, cat],
    ignore_index=True
)


OUT_DIR = Path("questionnaire_diagrams")
OUT_DIR.mkdir(exist_ok=True)


for questionnaire in sorted(df["score"].unique()):

    print(f"Processing {questionnaire}")

    qdf = df[df["score"] == questionnaire]


    for ftype in [
        "numeric",
        "binary",
        "categorical"
    ]:

        subset = qdf[qdf["type"] == ftype].copy()

        if subset.empty:
            continue


        # Clean invalid values
        subset["association"] = pd.to_numeric(
            subset["association"],
            errors="coerce"
        )

        subset = subset.replace(
            [np.inf, -np.inf],
            np.nan
        )

        subset = subset.dropna(
            subset=["association"]
        )

        if subset.empty:
            continue



        # Create labels
        if ftype == "categorical":

            subset["label"] = [
                textwrap.fill(
                    f"{row['factor']} = {row['value']}",
                    width=40
                )
                for _, row in subset.iterrows()
            ]

            subset = subset.sort_values(
                [
                    "factor",
                    "association"
                ]
            )

        else:

            subset = subset.sort_values(
                "association"
            )

            subset["label"] = [
                textwrap.fill(
                    str(label),
                    width=40
                )
                for label in subset["factor"]
            ]



        # Color mapping
        unique_factors = subset["factor"].unique()

        colors_cycle = (
            plt.rcParams["axes.prop_cycle"]
            .by_key()["color"]
        )

        color_map = {
            factor: colors_cycle[i % len(colors_cycle)]
            for i, factor in enumerate(unique_factors)
        }

        colors = [
            color_map[factor]
            for factor in subset["factor"]
        ]



        # Axis padding
        max_abs = subset["association"].abs().max()

        if pd.isna(max_abs) or max_abs == 0:
            max_abs = 1

        padding = max_abs * 0.3
        offset = max_abs * 0.03



        # Plot
        plt.figure(
            figsize=(
                14,
                max(8, len(subset) * 0.6)
            ),
            constrained_layout=True
        )


        bars = plt.barh(
            subset["label"],
            subset["association"],
            color=colors,
            alpha=0.7
        )


        plt.axvline(
            0,
            color="black",
            linewidth=1
        )


        # Extra room for text
        plt.xlim(
            subset["association"].min() - padding,
            subset["association"].max() + padding
        )


        # Add values
        for i, bar in enumerate(bars):

            value = subset["association"].iloc[i]
            n = int(subset["n"].iloc[i])


            if value >= 0:

                x_pos = value + offset
                alignment = "left"

            else:

                x_pos = value - offset
                alignment = "right"


            plt.text(
                x_pos,
                bar.get_y() + bar.get_height() / 2,
                f"{value:.3f} (n={n})",
                va="center",
                ha=alignment,
                fontsize=10,
                fontweight=(
                    "bold"
                    if abs(value) > 0.2
                    else "normal"
                ),
                clip_on=False
            )



        plt.title(
            f"{questionnaire.replace('_', ' ').title()} - "
            f"{ftype.capitalize()} Factors",
            fontsize=18,
            pad=20
        )


        plt.xlabel(
            "Spearman Correlation"
            if ftype == "numeric"
            else "Association (Mean Diff von globalem Durchschnitt)",
            fontsize=14
        )


        plt.grid(
            axis="x",
            alpha=0.2
        )


        # Space for long labels
        plt.subplots_adjust(
            left=0.35
        )


        plt.savefig(
            OUT_DIR / f"{questionnaire}_{ftype}.png",
            dpi=300,
            bbox_inches="tight"
        )


        plt.close()



print(f"\nDONE. Saved to: {OUT_DIR}")

Processing chiq_result


C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:270: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:270: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(


Processing dass_depression


C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:270: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppDat

Processing dass_fear


...

Processing midas_result


C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:270: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppDat

Processing pgic_result


C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:270: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(
C:\Users\veron\AppData\Local\Temp\ipykernel_33092\4129737313.py:101: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  subset = subset.replace(
C:\Users\veron\AppDat


DONE. Saved to: questionnaire_diagrams


In [13]:


from matplotlib.colors import to_rgb
# ============================================================
# COMBINED CATEGORICAL BAR CHART
# DASS Fear, Depression and Stress
# ============================================================


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

desired_scores = [
    "dass_fear",
    "dass_depression",
    "dass_stress"
]


labels = {
    "dass_fear": "Fear",
    "dass_depression": "Depression",
    "dass_stress": "Stress"
}


# ------------------------------------------------------------
# FILTER CATEGORICAL DATA
# ------------------------------------------------------------

cat_combined = df[
    df["type"] == "categorical"
].copy()


cat_combined = cat_combined[
    cat_combined["score"].isin(
        desired_scores
    )
].copy()


# ------------------------------------------------------------
# PIVOT DATA
# ------------------------------------------------------------

association_pivot = cat_combined.pivot_table(
    index=[
        "factor",
        "value"
    ],
    columns="score",
    values="association",
    aggfunc="first"
).reset_index()


# Sample sizes
n_pivot = cat_combined.pivot_table(
    index=[
        "factor",
        "value"
    ],
    columns="score",
    values="n",
    aggfunc="first"
).reset_index()


# Rename n columns
n_pivot = n_pivot.rename(
    columns={
        "dass_fear": "n_dass_fear",
        "dass_depression": "n_dass_depression",
        "dass_stress": "n_dass_stress"
    }
)


# Merge association and n values
pivot = association_pivot.merge(
    n_pivot,
    on=[
        "factor",
        "value"
    ],
    how="left"
)


# Make sure all three DASS columns exist
for score in desired_scores:

    if score not in pivot.columns:

        pivot[score] = np.nan


# ------------------------------------------------------------
# SORT DATA
# ------------------------------------------------------------

pivot = pivot.sort_values(
    [
        "factor",
        "value"
    ]
).reset_index(
    drop=True
)


# ------------------------------------------------------------
# CREATE LABELS
# ------------------------------------------------------------

pivot["label"] = [

    textwrap.fill(
        f"{factor} = {value}",
        width=40
    )

    for factor, value in zip(
        pivot["factor"],
        pivot["value"]
    )

]


# ============================================================
# COLOR MAPPING
# ============================================================


# Use the same default Matplotlib colors as above
colors_cycle = (

    plt.rcParams[
        "axes.prop_cycle"
    ]

    .by_key()
    ["color"]

)


# One base color for each categorical factor
unique_factors = pivot[
    "factor"
].unique()


color_map = {

    factor:
    colors_cycle[
        i % len(colors_cycle)
    ]

    for i, factor in enumerate(
        unique_factors
    )

}


from matplotlib.colors import to_rgb


def lighten_color(
    color,
    amount=0.55
):

    rgb = np.array(
        to_rgb(color)
    )

    return tuple(
        rgb
        + (
            1 - rgb
        )
        * amount
    )


def darken_color(
    color,
    amount=0.30
):

    rgb = np.array(
        to_rgb(color)
    )

    return tuple(
        rgb
        * (
            1 - amount
        )
    )


# ------------------------------------------------------------
# COLOR FOR EACH DASS SCALE
# ------------------------------------------------------------

colors = {

    "dass_fear": [],

    "dass_depression": [],

    "dass_stress": []

}


for factor in pivot[
    "factor"
]:

    base_color = color_map[
        factor
    ]


    # Fear = light
    colors[
        "dass_fear"
    ].append(

        lighten_color(
            base_color,
            amount=0.55
        )

    )


    # Depression = medium/base
    colors[
        "dass_depression"
    ].append(

        base_color

    )


    # Stress = dark
    colors[
        "dass_stress"
    ].append(

        darken_color(
            base_color,
            amount=0.30
        )

    )


# ============================================================
# PLOT SETTINGS
# ============================================================


# Thin bars
bar_height = 0.22


# No gap between the three bars
# of the same categorical value
#
# Additional space between different
# categorical values
group_height = (
    bar_height
    * 3
    + 0.20
)


y = (

    np.arange(
        len(pivot)
    )

    * group_height

)


# Positions of the three bars
#
# The bars touch each other:
#
# Fear
# Depression
# Stress
#

positions = {

    "dass_fear":
    y + bar_height,

    "dass_depression":
    y,

    "dass_stress":
    y - bar_height

}


# ============================================================
# CREATE FIGURE
# ============================================================


plt.figure(

    figsize=(

        16,

        max(

            10,

            len(pivot)
            * 0.9

        )

    ),

    constrained_layout=True

)


# ============================================================
# DRAW BARS
# ============================================================


for score in desired_scores:

    plt.barh(

        positions[
            score
        ],

        pivot[
            score
        ],

        height=bar_height,

        color=colors[
            score
        ],

        label=labels[
            score
        ]

    )


# ------------------------------------------------------------
# ZERO LINE
# ------------------------------------------------------------

plt.axvline(

    0,

    color="black",

    linewidth=1

)


# ------------------------------------------------------------
# Y-AXIS LABELS
# ------------------------------------------------------------

plt.yticks(

    y,

    pivot[
        "label"
    ]

)


# ============================================================
# AXIS LIMITS
# ============================================================


all_values = (

    pivot[
        desired_scores
    ]

    .values

    .flatten()

)


all_values = all_values[
    ~np.isnan(
        all_values
    )
]


if len(
    all_values
) > 0:


    max_abs = np.max(

        np.abs(
            all_values
        )

    )


    if max_abs == 0:

        max_abs = 1


    padding = (

        max_abs
        * 0.30

    )


    plt.xlim(

        np.min(
            all_values
        )
        - padding,


        np.max(
            all_values
        )
        + padding

    )


else:

    max_abs = 1


# ============================================================
# VALUE LABELS
# ============================================================

for i, row in pivot.iterrows():

    for score in desired_scores:

        value = row[score]

        if pd.isna(value):

            continue


        y_position = positions[
            score
        ][i]


        # Show n only for the middle bar:
        # dass_depression
        if score == "dass_depression":

            n = row[
                "n_dass_depression"
            ]

            label = (
                f"{value:.3f} "
                f"(n={int(n)})"
            )

        else:

            label = (
                f"{value:.3f}"
            )


        if value >= 0:

            x_position = (
                value
                + max_abs * 0.03
            )

            alignment = "left"

        else:

            x_position = (
                value
                - max_abs * 0.03
            )

            alignment = "right"


        plt.text(

            x_position,

            y_position,

            label,

            va="center",

            ha=alignment,

            fontsize=9

        )

# ============================================================
# FORMATTING
# ============================================================


plt.title(

    "Categorical Factors: "
    "Fear, Depression and Stress",

    fontsize=18,

    pad=20

)


plt.xlabel(

    "Association "
    "(Mean Diff von globalem Durchschnitt)",

    fontsize=14

)


plt.grid(

    axis="x",

    alpha=0.2

)


plt.legend(

    title="Questionnaire"

)


# Space for long labels
plt.subplots_adjust(

    left=0.35

)


# ============================================================
# SAVE FIGURE
# ============================================================


output_file = (

    OUT_DIR
    / "categorical_combined_"
      "fear_depression_stress.png"

)


plt.savefig(

    output_file,

    dpi=300,

    bbox_inches="tight"

)


plt.close()


print(

    "\nCombined categorical bar chart saved to:"

)


print(

    output_file

)

C:\Users\veron\AppData\Local\Temp\ipykernel_33092\2543449057.py:601: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  plt.subplots_adjust(



Combined categorical bar chart saved to:
questionnaire_diagrams\categorical_combined_fear_depression_stress.png
